In [52]:
import os 
from typing import TypedDict
from langchain.chat_models import init_chat_model


# langgraph
from langgraph.graph import START,END,StateGraph


# for Rag
from langchain_community.document_loaders import WebBaseLoader,PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings

# vector
from langchain_community.vectorstores import FAISS


from dotenv import load_dotenv

load_dotenv()
os.environ[("OPENAI_API_KEY")]=os.getenv("OPENAI_API_KEY")

model=init_chat_model("gpt-4o")

In [38]:
# load the documents from the Documents folder 
documents=[]
folder_path="./Documents"

def load_folder_pdf(folder_path:str):
    all_doc=[]
    for filename in os.listdir(folder_path):
        if filename.endswith(".pdf"):
            file_path=os.path.join(folder_path,filename)
            print(f"Loading: {filename}")
            
            loader=PyPDFLoader(file_path)
            docs=loader.load()
            all_doc.extend(docs)
    
    return all_doc

documents=load_folder_pdf(folder_path)
print(f"Loaded {len(documents)} documents")

Loading: Code_Agent_Final_Roadmap.pdf.pdf
Loading: DS -Intern Assignment .pdf
Loaded 37 documents


In [39]:
# split the doc into chunks
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

chunks=text_splitter.split_documents(documents)

In [ ]:
# init the embedding modle 
embedding=OpenAIEmbeddings(model="text-embedding-3-small")
vector_store=FAISS.from_documents(chunks,embedding)

# retriver
retriver=vector_store.as_retriever(search_type="similarity",search_kwargs={"k":4})

In [53]:
class App_State(TypedDict):
    user_question:str
    answer:str

In [ ]:
# build the graph
graph=StateGraph(App_state)
graph.add_Node("Retrive")
graph.add_Node("Generate")

graph.add_edge(START,"Retrive")
graph.add_edge("Generate",END)

# compile the graph
workflow=graph.compile()